In [2]:
import os 
from dotenv import load_dotenv

if os.environ.get("API_KEY") is None:
    raise ValueError("API_KEY environment variable is not set. Please set it in your .env file.")

else:
    print("API_KEY environment variable is set.")

API_KEY environment variable is set.


Test API

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model= "gpt-5-nano", temperature=0, api_key = os.environ.get("API_KEY"))

response = llm.invoke("Hello what is the capital of France?")

print(response.content)

Paris. Would you like a bit more info about the city or France?


### RAG IMPLEMENTATION WITH  ADDED PDFS 

### Step 1: Extracting Text From PDFS

In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_admin_path = "/Users/jessepepple/Downloads/RAG_Project_Main/Docs/fabric-admin.pdf"

Loader = PyPDFLoader(pdf_admin_path)

docs = Loader.load()

/var/folders/90/v950rdks7f93s0k0r2wr6kfh0000gn/T/ipykernel_1937/1256869173.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Creating our Metadata

In [5]:
for i in docs:
    i.metadata = {
        "source" : "fabric-admin.pdf",
        "developer" : "Microsoft"
    }

In [6]:
docs[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}

### Step 2 Chunking

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅ OVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂ GET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃ HOW-TO GUIDE\nUnderstand Fabric admin roles\nｉ REFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅ OVERVIEW\nAbout tenant settings\nｃ HOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Configure notifications\nSet up metadata scanning\nEnable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅ OVERVIEW\nWhat is the admin monitoring workspace?\

### Step 3 Create Embeddings 

In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model= "text-embedding-3-small", api_key = os.environ.get("API_KEY"))
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x11cf8c590>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x11cf8c830>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

### Step 4 Create and Store Embeddings In The Existing Local Vector Store

In [9]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(persist_directory="/Users/jessepepple/Downloads/RAG_Project_Main/Vector", embedding_function=embedding_model)

/var/folders/90/v950rdks7f93s0k0r2wr6kfh0000gn/T/ipykernel_1937/441968580.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory="/Users/jessepepple/Downloads/RAG_Project_Main/Vector", embedding_function=embedding_model)


In [10]:
vectorstore.add_documents(chunks)

['5b054281-cafe-446c-b126-0ccd27a8f255',
 'f77f8d70-58cd-451a-bcbe-51f8d7960870',
 '57abd864-acc6-4b1d-bd2c-2ddd55dae402',
 '5db2bb2a-2fcb-45ea-b0d0-4a1f963c5a1c',
 '1806dd5c-8f4b-45b7-93cf-6e48e673c352',
 '56b80b00-44c3-440f-be2e-1776467cbdc6',
 'a9254e87-18db-4eb3-b9a5-86cf22592866',
 '553d9ee1-6b8f-4510-b7c3-296163db4465',
 '1a43a14f-5057-41e2-8fd4-f36d44902240',
 'fd64bfa3-af8e-47ad-a8d0-ce87c39a5dbc',
 '209243cf-46cc-4bef-a929-77ff3e137bc0',
 'd513d0e2-c548-4036-8cff-67e6d3484063',
 '0ad125d1-4a5b-4c3d-8461-4162509768d0',
 'a16621f8-318c-442f-aaae-cf17effa4ad4',
 'c95ece6b-0415-4e4c-95eb-8213c34bf970',
 '5af5bafb-9a2a-4e9f-b438-44bbac8c2646',
 '5056b8bd-18bb-4961-bc53-6a0d1981ac1b',
 '1014750d-d996-4ef9-a2e7-a70611fdc561',
 '37ecffb9-b2db-464f-9a11-5a624a10352d',
 'd061b0c9-399b-4824-ba1d-0714d321c5e4',
 'e3eccd51-7973-499c-af9a-ff7feff7d6d8',
 'dbbd5592-08c3-43d8-ad8a-d14d95248160',
 '13d2de08-c46b-4f77-a7c0-511e755b1d87',
 'd9ba84f5-e993-46ea-9b2b-a4bd026f7950',
 '22a1a481-3e63-

### Step 5: Semantic Search 

In [11]:
vectorstore.similarity_search("What is OneLake?", k=3)

[Document(metadata={'developer': 'Microsoft', 'source': 'fabric-onelake.pdf'}, page_content='OneLake, the OneDrive for data\nArticle • 07/25/2024\nOneLake is a single, unified, logical data lake for your whole organization. A data Lake\nprocesses large volumes of data from various sources. Like OneDrive, OneLake comes\nautomatically with every Microsoft Fabric tenant and is designed to be the single place\nfor all your analytics data. OneLake brings customers:\nOne data lake for the entire organization\nOne copy of data for use with multiple analytical engines\nBefore OneLake, it was easier for customers to create multiple lakes for different\nbusiness groups rather than collaborating on a single lake, even with the extra overhead\nof managing multiple resources. OneLake focuses on removing these challenges by\nimproving collaboration. Every customer tenant has exactly one OneLake. There can\nnever be more than one and if you have Fabric, there can never be zero. Every Fabric\ntenant a

Talk To LLM

In [12]:
context = vectorstore.similarity_search("What is Microsoft Fabric Admin?", k=3)
context

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='What is Microsoft Fabric admin?\n07/01/2024\nMicrosoft Fabric admin is the management of the organization-wide settings that control how\nMicrosoft Fabric works. Users that are assigned to admin roles configure, monitor, and\nprovision organizational resources. This article provides an overview of admin roles, tasks, and\ntools to help you get started.\nThere are several roles that work together to administer Microsoft Fabric for your organization.\nMost admin roles are assigned in the Microsoft 365 admin portal or by using PowerShell. The\ncapacity admin roles are assigned when the capacity is created. To learn more about each of\nthe admin roles, see About admin roles. To learn how to assign admin roles, see Assign admin\nroles.\nThis section lists the Microsoft 365 admin roles and the tasks they can perform.\nGlobal administrator\nUnlimited access to all management features for the organizatio

In [13]:
response  = llm.invoke(f"Answer the question: regarding Microsoft Fabric Admin: {context}")
print(response.content)

Here’s a concise overview of Microsoft Fabric Admin based on the provided document content:

- What it is
  - Microsoft Fabric Admin is the management layer for organization-wide settings that control how Microsoft Fabric works. Admins configure, monitor, and provision organizational resources to govern the platform.

- How admin work is organized
  - Admin roles exist to share responsibilities and can be assigned to users to perform specific tasks.
  - Most admin roles are assigned in the Microsoft 365 admin portal or by using PowerShell. Capacity admin roles are assigned when you create a capacity.

- Example roles and capabilities
  - Global administrator: Unlimited access to all management features for the organization; can assign roles to other users.
  - Billing administrator: Manage subscriptions and purchase licenses.

- How to learn more
  - To learn about each admin role, see “About admin roles” and to learn how to assign admin roles, see “Assign admin roles.”
  - The article